# M2SVid full Colab inference notebook

Complete inference pipeline: input MP4 → DepthCrafter depth → depth-based warping → M2SVid refinement → display/archive outputs.

Recommended GPU: **L4 24GB or A100**. Start with official demo; later switch to a short custom MP4 in Drive.

## 0. Mount Drive and check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi
!python --version
!df -h /content /content/drive

## 1. Paths and experiment settings

In [ ]:
from pathlib import Path
import os, time, shutil, subprocess, json
DRIVE_ROOT = Path('/content/drive/MyDrive/m2svid_full_inference')
CKPT_DIR = DRIVE_ROOT / 'ckpts'
INPUT_DIR = DRIVE_ROOT / 'inputs'
OUTPUT_ARCHIVE = DRIVE_ROOT / 'outputs'
LOG_DIR = DRIVE_ROOT / 'logs'
for p in [DRIVE_ROOT, CKPT_DIR, INPUT_DIR, OUTPUT_ARCHIVE, LOG_DIR]: p.mkdir(parents=True, exist_ok=True)
USE_OFFICIAL_DEMO = True
CUSTOM_INPUT_VIDEO = INPUT_DIR / 'my_clip.mp4'
MAX_RES = 512
DEPTH_STEPS = 25
DISPARITY_PERC = 0.05
EXP_NAME = time.strftime('full_demo_%Y%m%d_%H%M%S')
print('Drive root:', DRIVE_ROOT)
print('Experiment:', EXP_NAME)

## 2. Clone M2SVid repo

In [ ]:
%cd /content
if not Path('/content/m2svid').exists():
    !git clone --depth 1 https://github.com/google-research/m2svid.git /content/m2svid
%cd /content/m2svid
!git rev-parse --short HEAD
!find demo -maxdepth 3 -type f | sort

## 3. Download/link M2SVid and Hi3D checkpoints

In [ ]:
%cd /content/m2svid
!pip -q install gdown
if not (CKPT_DIR / 'm2svid_weights.pt').exists():
    !wget -c -O {CKPT_DIR/'m2svid_weights.pt'} https://storage.googleapis.com/gresearch/m2svid/m2svid_weights.pt
else:
    print('M2SVid checkpoint already exists in Drive')
hi3d_zip = CKPT_DIR / 'hi3d_ckpts.zip'
if not hi3d_zip.exists() and not (CKPT_DIR / 'open_clip_pytorch_model.bin').exists():
    !gdown --fuzzy 'https://drive.google.com/file/d/1j_NEG2CPhFeRetYziWK6Qe62R5h7lG_V/view?usp=sharing' -O {hi3d_zip}
else:
    print('Hi3D zip or open_clip checkpoint already exists')
!mkdir -p /content/m2svid/ckpts
if hi3d_zip.exists() and not Path('/content/m2svid/ckpts/open_clip_pytorch_model.bin').exists():
    !unzip -n {hi3d_zip} -d /content/m2svid/
if (CKPT_DIR / 'open_clip_pytorch_model.bin').exists():
    !ln -sf {CKPT_DIR/'open_clip_pytorch_model.bin'} /content/m2svid/ckpts/open_clip_pytorch_model.bin
!ln -sf {CKPT_DIR/'m2svid_weights.pt'} /content/m2svid/ckpts/m2svid_weights.pt
!find /content/m2svid/ckpts -maxdepth 2 -type f | sort | head -80

## 4. Install M2SVid/SGM dependencies

In [ ]:
%cd /content/m2svid
!pip -q install omegaconf einops pytorch-lightning==1.5.0 kornia==0.6.9 open-clip-torch==2.29.0 pytorch-msssim==1.0.0 ffmpeg-python av imageio safetensors xformers || true
!python - <<'PY'
import torch, torchvision
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('torchvision', torchvision.__version__)
PY

## 5. Install/prepare DepthCrafter

In [ ]:
%cd /content/m2svid
!mkdir -p third_party
if not Path('third_party/DepthCrafter').exists():
    !git clone --depth 1 https://github.com/Tencent/DepthCrafter.git third_party/DepthCrafter
%cd /content/m2svid/third_party/DepthCrafter
if Path('requirements.txt').exists():
    !pip -q install -r requirements.txt || true
else:
    !pip -q install diffusers transformers accelerate decord opencv-python imageio imageio-ffmpeg scipy matplotlib || true
%cd /content/m2svid
!test -f third_party/DepthCrafter/run.py && echo 'DepthCrafter run.py found' || (echo 'DepthCrafter run.py missing'; find third_party/DepthCrafter -maxdepth 2 -type f | head -50)

## 6. Prepare input video

In [ ]:
%cd /content/m2svid
if USE_OFFICIAL_DEMO:
    INPUT_VIDEO = Path('/content/m2svid/demo/input.mp4')
else:
    INPUT_VIDEO = Path(CUSTOM_INPUT_VIDEO)
    assert INPUT_VIDEO.exists(), f'Custom input missing: {INPUT_VIDEO}'
WORK_INPUT = Path('/content/m2svid/inputs/input.mp4')
WORK_INPUT.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(INPUT_VIDEO, WORK_INPUT)
print('Using input:', INPUT_VIDEO)
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate,duration -of default=nw=1 {WORK_INPUT}

## 7. Run DepthCrafter depth prediction

In [ ]:
%cd /content/m2svid
!mkdir -p outputs/depthcrafter_full
start = time.time()
cmd = f'''PYTHONPATH="third_party/DepthCrafter/::${{PYTHONPATH}}" python third_party/DepthCrafter/run.py --video-path {WORK_INPUT} --save_folder outputs/depthcrafter_full --save_npz True --num_inference_steps {DEPTH_STEPS} --max_res {MAX_RES}'''
print(cmd)
ret = subprocess.run(cmd, shell=True)
print('return code:', ret.returncode, 'elapsed sec:', round(time.time() - start, 1))
!find outputs/depthcrafter_full -maxdepth 2 -type f -print
!nvidia-smi

## 8. Locate depth `.npz`

In [ ]:
candidates = sorted(Path('/content/m2svid/outputs/depthcrafter_full').glob('*.npz'))
print(candidates)
assert candidates, 'No depth .npz found. Check DepthCrafter output above.'
DEPTH_NPZ = candidates[0]
print('Using depth:', DEPTH_NPZ)

## 9. Run depth-based warping

In [ ]:
%cd /content/m2svid
!mkdir -p outputs/reprojected_full
start = time.time()
cmd = f'''PYTHONPATH="./:./third_party/Hi3D-Official/:./third_party/pytorch-msssim/:${{PYTHONPATH}}" python warping.py --video_path {WORK_INPUT} --depth_path {DEPTH_NPZ} --output_path_reprojected outputs/reprojected_full/input_reprojected.mp4 --output_path_mask outputs/reprojected_full/input_reprojected_mask.mp4 --disparity_perc {DISPARITY_PERC}'''
print(cmd)
ret = subprocess.run(cmd, shell=True)
print('return code:', ret.returncode, 'elapsed sec:', round(time.time() - start, 1))
!find outputs/reprojected_full -maxdepth 2 -type f -print

## 10. Run M2SVid inpainting/refinement

In [ ]:
%cd /content/m2svid
!mkdir -p outputs/m2svid_full
start = time.time()
cmd = f'''PYTHONPATH="./:./third_party/Hi3D-Official/:./third_party/pytorch-msssim/:${{PYTHONPATH}}" python inpaint_and_refine.py --mask_antialias 0 --model_config configs/m2svid.yaml --ckpt ckpts/m2svid_weights.pt --video_path {WORK_INPUT} --reprojected_path outputs/reprojected_full/input_reprojected.mp4 --reprojected_mask_path outputs/reprojected_full/input_reprojected_mask.mp4 --output_folder outputs/m2svid_full'''
print(cmd)
ret = subprocess.run(cmd, shell=True)
print('return code:', ret.returncode, 'elapsed sec:', round(time.time() - start, 1))
!find outputs/m2svid_full -maxdepth 2 -type f -print
!nvidia-smi

## 11. Display results

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
def show_video(path, width=720):
    path = Path(path); assert path.exists(), f'Missing {path}'
    data = 'data:video/mp4;base64,' + b64encode(path.read_bytes()).decode()
    display(HTML(f'<video width="{width}" controls loop><source src="{data}" type="video/mp4"></video><p><code>{path}</code></p>'))
print('Input'); show_video(WORK_INPUT, 520)
print('Warped right'); show_video('/content/m2svid/outputs/reprojected_full/input_reprojected.mp4', 520)
print('Mask'); show_video('/content/m2svid/outputs/reprojected_full/input_reprojected_mask.mp4', 520)
print('M2SVid side-by-side'); show_video('/content/m2svid/outputs/m2svid_full/input_sbs.mp4', 720)

## 12. Archive outputs to Drive

In [ ]:
archive = OUTPUT_ARCHIVE / EXP_NAME
archive.mkdir(parents=True, exist_ok=True)
for rel in ['inputs', 'outputs/depthcrafter_full', 'outputs/reprojected_full', 'outputs/m2svid_full']:
    src = Path('/content/m2svid') / rel
    if src.exists():
        dst = archive / rel.replace('/', '_')
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(src, dst)
meta = {'exp_name': EXP_NAME, 'use_official_demo': USE_OFFICIAL_DEMO, 'input_video': str(INPUT_VIDEO), 'max_res': MAX_RES, 'depth_steps': DEPTH_STEPS, 'disparity_perc': DISPARITY_PERC}
(archive / 'meta.json').write_text(json.dumps(meta, indent=2))
print('Archived to:', archive)
!find {archive} -maxdepth 2 -type f | sort | head -100